In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ast

In [ ]:
mean_file = "/jdata/qmy/VirtualCell/diff_matrix_predict/drug_cell_diff_5.0uM.csv"
pseudo_file = "/jdata/qmy/VirtualCell/diff_matrix_predict/sc_pseudo_bulk_10_diff_5.0uM.csv"
pred_file = "/jdata/qmy/VirtualCell/diff_matrix_predict/sc_predictions_5.0uM.csv"

mean_df = pd.read_csv(mean_file, index_col=[0,1])
pseudo_df = pd.read_csv(pseudo_file, index_col=[0,1])
pred_df = pd.read_csv(pred_file,index_col=[0,1])

In [ ]:
print(f"mean shape: {mean_df.shape}")
print(f"pseudo shape: {pseudo_df.shape}")
print(f"pred shape: {pred_df.shape}")
common_samples = mean_df.index.intersection(pseudo_df.index).intersection(pred_df.index)
print(f"共同样本数: {len(common_samples)}")

In [ ]:
conc = 5.0
conc_str = str(conc)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
all_points = []
labels = []
for idx in common_samples:
    all_points.append(mean_df.loc[idx].values)
    labels.append(f"{idx[0]}_{idx[1]}_mean")
    all_points.append(pseudo_df.loc[idx].values)
    labels.append(f"{idx[0]}_{idx[1]}_sc")
    all_points.append(pred_df.loc[idx].values)
    labels.append(f"{idx[0]}_{idx[1]}_pred")

all_points = np.array(all_points)
pca = PCA(n_components=2)
coords = pca.fit_transform(all_points)
on_line_count = 0
total_pred = 0
distances = []
for idx in common_samples:
    mean_idx = labels.index(f"{idx[0]}_{idx[1]}_mean")
    sc_idx = labels.index(f"{idx[0]}_{idx[1]}_sc")
    pred_idx = labels.index(f"{idx[0]}_{idx[1]}_pred")
    
    mean_pt = coords[mean_idx]
    sc_pt = coords[sc_idx]
    pred_pt = coords[pred_idx]

    vec = sc_pt - mean_pt
    w = pred_pt - mean_pt
    
    if np.linalg.norm(vec) > 1e-10:
        t = np.dot(w, vec) / np.dot(vec, vec)
        proj = mean_pt + t * vec
        dist = np.linalg.norm(pred_pt - proj)
        distances.append(dist)
        if 0 <= t <= 1 and dist < 0.5:
            on_line_count += 1
    else:
        distances.append(0)
    total_pred += 1


In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
colors = {'mean': 'blue', 'sc': 'green', 'pred': 'red'}
markers = {'mean': 'o', 'sc': 's', 'pred': '^'}
for i, label in enumerate(labels):
    if '_mean' in label:
        color, marker, size = 'blue', 'o', 15
    elif '_sc' in label:
        color, marker, size = 'green', 's', 15
    else:
        color, marker, size = 'red', '^', 12
    
    ax.scatter(coords[i, 0], coords[i, 1], c=color, marker=marker, alpha=0.6, s=size)

# 连接每个样本的三个点
for idx in common_samples:
    points = []
    for suffix in ['_mean', '_sc', '_pred']:
        label = f"{idx[0]}_{idx[1]}{suffix}"
        if label in labels:
            pos = labels.index(label)
            points.append(coords[pos])
    
    if len(points) == 3:
        points = np.array(points)
        ax.plot(points[:, 0], points[:, 1], 'k-', alpha=0.2, linewidth=0.5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
ax.set_title(f'PCA: Mean, Single-cell, Prediction (0.5uM)\nPred on line: {on_line_count}/{total_pred} ({on_line_count/total_pred*100:.1f}%)')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=8, label='Mean'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='green', markersize=8, label='Single-cell'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='red', markersize=8, label='Prediction'),
    Line2D([0], [0], color='black', linewidth=1, label='Mean-SC line')
]
ax.legend(handles=legend_elements, loc='best')
ax.grid(True, alpha=0.3)

plt.savefig('/jdata/qmy/VirtualCell/pca_mean_sc_pred_5.0uM.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"总预测样本数: {total_pred}")
print(f"在mean-sc连线上的预测样本数: {on_line_count}")
print(f"比例: {on_line_count/total_pred*100:.1f}%")
print(f"平均垂直距离: {np.mean(distances):.4f}")